<a href="https://colab.research.google.com/github/NghiaH19/E-commerce/blob/main/KNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
import joblib

# 1. Kết nối Google Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Tải dữ liệu lịch sử đã làm sạch
# Đảm bảo đường dẫn này khớp với thư mục của bạn trên Drive
file_path = '/content/drive/MyDrive/Project2_Ecommerce/ecommerce_final_processed.csv'
df = pd.read_csv(file_path)

print(f"✅ Đã tải thành công dữ liệu với {len(df):,} dòng.")

Mounted at /content/drive
✅ Đã tải thành công dữ liệu với 2,662,922 dòng.


In [ ]:
product_lookup = df.groupby('product_id').agg({
    'category_code': 'first',
    'brand': 'first',
    'price': 'mean',
    'luot_view': 'mean',
    'luot_mua': 'mean'
}).reset_index()

print(f"✅ Đã nén thành công {len(product_lookup):,} sản phẩm duy nhất trong kho.")
display(product_lookup.head())

✅ Đã nén thành công 66,538 sản phẩm duy nhất trong kho.


,product_id,category_code,brand,price,luot_view,luot_mua
0,1000978.0,98,1487,314.451176,245.647059,0.735294
1,1001588.0,98,1118,127.859630,63.962963,0.259259
2,1001606.0,98,97,363.070000,0.562500,0.000000
3,1001618.0,98,97,522.136000,25.400000,0.000000
4,1002042.0,98,1487,77.140000,1.537037,0.000000


In [ ]:
scaler_knn = StandardScaler()

# Chỉ scale 3 cột đặc trưng đầu vào (không đưa lượt mua/xem vào vì đó là cái cần dự báo)
features_to_scale = ['category_code', 'brand', 'price']
product_features_scaled = scaler_knn.fit_transform(product_lookup[features_to_scale])

# Lưu bộ chuẩn hóa để sau này Web App dùng lại
joblib.dump(scaler_knn, '/content/drive/MyDrive/Project2_Ecommerce/scaler_knn.pkl')

print("✅ Đã chuẩn hóa xong 3 đặc trưng (Category, Brand, Price).")
print("✅ Đã lưu file: scaler_knn.pkl")

✅ Đã chuẩn hóa xong 3 đặc trưng (Category, Brand, Price).
✅ Đã lưu file: scaler_knn.pkl


In [ ]:
model_knn = NearestNeighbors(n_neighbors=5, metric='euclidean')

# Dạy cho AI bản đồ các sản phẩm (đã chuẩn hóa)
model_knn.fit(product_features_scaled)

# Lưu lại mô hình và bảng tra cứu
joblib.dump(model_knn, '/content/drive/MyDrive/Project2_Ecommerce/knn_model.pkl')
joblib.dump(product_lookup, '/content/drive/MyDrive/Project2_Ecommerce/product_lookup.pkl')

print("✅ Đã huấn luyện xong mô hình KNN!")
print("✅ Đã lưu file: knn_model.pkl")
print("✅ Đã lưu file: product_lookup.pkl")

✅ Đã huấn luyện xong mô hình KNN!
✅ Đã lưu file: knn_model.pkl
✅ Đã lưu file: product_lookup.pkl


In [3]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Đang tải cơ sở dữ liệu sản phẩm...")
# Tải lại file product_lookup bạn đã tạo ở bước trước
product_lookup = joblib.load('/content/drive/MyDrive/Project2_Ecommerce/product_lookup.pkl')

# 1. Kịch bản kiểm thử: Giả lập Khởi động lạnh
# Tách 20% sản phẩm ra làm "Sản phẩm mới" (Test set), 80% làm "Kho hàng cũ" (Train set)
train_df, test_df = train_test_split(product_lookup, test_size=0.2, random_state=42)

features = ['category_code', 'brand', 'price']
target = 'luot_mua'

# 2. Chuẩn hóa không gian vector dựa trên tập Train
scaler_eval = StandardScaler()
X_train_scaled = scaler_eval.fit_transform(train_df[features])
X_test_scaled = scaler_eval.transform(test_df[features]) # Ẩn thông tin, chỉ scale theo Train

# 3. Huấn luyện KNN trên "Kho hàng cũ"
knn_eval = NearestNeighbors(n_neighbors=5, metric='euclidean')
knn_eval.fit(X_train_scaled)

# 4. Bắt đầu dự báo cho tập "Sản phẩm mới"
print("Đang cho AI quét tìm hàng xóm và dự phóng doanh số...")
distances, indices = knn_eval.kneighbors(X_test_scaled)

y_pred_knn = []
# Duyệt qua từng sản phẩm mới
for neighbor_indices in indices:
    # Lấy doanh số thực tế của 5 hàng xóm trong kho cũ
    neighbor_sales = train_df.iloc[neighbor_indices][target]
    # Doanh số dự phóng = Trung bình doanh số của 5 hàng xóm
    y_pred_knn.append(neighbor_sales.mean())

y_test_knn = test_df[target].values

# 5. Đánh giá sai số
mae_knn = mean_absolute_error(y_test_knn, y_pred_knn)
rmse_knn = np.sqrt(mean_squared_error(y_test_knn, y_pred_knn))
r2_knn = r2_score(y_test_knn, y_pred_knn)

print(f"\n--- 🎯 KẾT QUẢ ĐÁNH GIÁ KNN (COLD-START) ---")
print(f"Tổng số sản phẩm giả lập test: {len(test_df):,} mã hàng")
print(f"✅ MAE  : {mae_knn:.4f} (Lệch trung bình {mae_knn:.2f} đơn/ngày)")
print(f"✅ RMSE : {rmse_knn:.4f}")
print(f"✅ R2   : {r2_knn:.4f}")

Đang tải cơ sở dữ liệu sản phẩm...
Đang cho AI quét tìm hàng xóm và dự phóng doanh số...

--- 🎯 KẾT QUẢ ĐÁNH GIÁ KNN (COLD-START) ---
Tổng số sản phẩm giả lập test: 13,308 mã hàng
✅ MAE  : 0.5087 (Lệch trung bình 0.51 đơn/ngày)
✅ RMSE : 7.1274
✅ R2   : -0.2809
